# 07 — Public transport: GTFS import and the transit database

AequilibraE models transit from **GTFS feeds** (the de-facto standard for transit
schedules). Importing a feed creates `public_transport.sqlite` with routes, patterns,
stops, trips and — after map-matching — the real paths through the road network.

The Coquimbo example ships with a GTFS feed for the *Lisanco* operator, which we
import from scratch.


In [1]:
from os import remove
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.transit import Transit
from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "coquimbo")

# The example ships with a transit DB already built - remove it so we import cleanly
remove(str(Path(fldr) / "public_transport.sqlite"))

In [2]:
data = Transit(project)

gtfs = data.new_gtfs_builder(agency="Lisanco", file_path=str(Path(fldr) / "gtfs_coquimbo.zip"))

# A GTFS feed is a schedule over many days: pick the service day to import.
gtfs.load_date("2016-04-13")

# Map-matching (finding the true road path for each pattern) is optional and slower:
# gtfs.set_allow_map_match(True); gtfs.map_match()

gtfs.save_to_disk()

Loading routes (Step: 1/12)                       : 0it [00:00, ?it/s]

Loading stops (Step: 2/12)                        :   0%|          | 0/78 [00:00<?, ?it/s]

Loading stop times (Step: 3/12)                   :   0%|          | 0/1007 [00:00<?, ?it/s]

Loading shapes (Step: 4/12)                       :   0%|          | 0/2 [00:00<?, ?it/s]

Loading trips (Step: 5/12)                        :   0%|          | 0/1007 [00:00<?, ?it/s]

...

Loading data for 2016-04-13 (Step: 9/12) -        :   0%|          | 0/1 [00:00<?, ?it/s]

Saving patterns (Step: 10/12)                     :   0%|          | 0/2 [00:00<?, ?it/s]

Saving trips (Step: 11/12)                        :   0%|          | 0/360 [00:00<?, ?it/s]

Saving links (Step: 11/12)                        :   0%|          | 0/78 [00:00<?, ?it/s]

Saving stops (Step: 12/12)                        :   0%|          | 0/78 [00:00<?, ?it/s]

In [3]:
import geopandas as gpd
import pandas as pd

with project.transit_connection as conn:
    routes = pd.read_sql("SELECT route_id, route, ST_AsText(geometry) wkt FROM routes", conn)
    stops = pd.read_sql("SELECT stop_id, ST_X(geometry) x, ST_Y(geometry) y FROM stops", conn)
    trips = pd.read_sql("SELECT count(*) n FROM trips", conn)

routes_gdf = gpd.GeoDataFrame(routes.drop(columns="wkt"),
                              geometry=gpd.GeoSeries.from_wkt(routes["wkt"]), crs=4326)
stops_gdf = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.x, stops.y), crs=4326)

print(f"{len(routes_gdf)} routes, {len(stops_gdf)} stops, {trips.n[0]} trips imported")

2 routes, 78 stops, 360 trips imported


In [4]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
#
# add_gdf also translates the declarative symbology into the OpenLayers
# flat-style expressions the current JupyterGIS frontend renders from, so
# colours and line widths show up without touching the symbology panel.
import json

import matplotlib.colors
import matplotlib.pyplot as _plt
from jupytergis import GISDocument
from jupytergis_lab.notebook.symbology import to_symbology_state

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def _hex(rgba):
    return matplotlib.colors.to_hex(rgba)

def _ramp_expr(fld, params):
    name, dom = params.get("name", "viridis"), params.get("domain") or [0.0, 1.0]
    cmap = _plt.get_cmap(name)
    if params.get("reverse"):
        cmap = cmap.reversed()
    expr = ["interpolate", ["linear"], ["get", fld]]
    for i in range(7):
        t = i / 6
        expr += [dom[0] + t * (dom[1] - dom[0]), _hex(cmap(t))]
    return expr

def _scalar_expr(fld, params):
    d, r = params["domain"], params["range"]
    return ["interpolate", ["linear"], ["get", fld], d[0], r[0], d[1], r[1]]

def _cat_expr(fld, params, gdf):
    cmap = _plt.get_cmap(params.get("colorRamp", "tab10"))
    vals = list(dict.fromkeys(gdf[fld].dropna()))
    expr = ["match", ["get", fld]]
    for i, v in enumerate(vals):
        expr += [v, _hex(cmap(i % cmap.N))]
    return expr + ["#9ca3af"]

def _flat_style(symbology, gdf):
    """Grammar symbology -> OpenLayers flat-style dict (what the map renders)."""
    state = to_symbology_state(symbology)
    if not state:
        return None
    flat = {}
    for layer in state.get("layers", []):
        for rule in layer.get("rules", []):
            flds = rule.get("fields") or [None]
            for m in rule.get("mappings", []):
                scheme = m["scale"]["scheme"]
                params = m["scale"].get("params", {})
                if scheme == "constant_rgba":
                    val = params["value"]
                    val = _hex([c if c <= 1 else c / 255 for c in val]) if isinstance(val, (list, tuple)) else val
                elif scheme == "constant_num":
                    val = params["value"]
                elif scheme == "colorMap":
                    val = _ramp_expr(flds[0], params)
                elif scheme == "scalar":
                    val = _scalar_expr(flds[0], params)
                elif scheme == "categorical":
                    val = _cat_expr(flds[0], params, gdf)
                else:
                    continue
                for enc in m.get("encodings", []):
                    flat[enc] = val
    if any(k.startswith("circle") for k in flat) and "circle-radius" not in flat:
        flat["circle-radius"] = 5
    if "stroke-color" in flat and "stroke-width" not in flat:
        flat["stroke-width"] = 1.5
    return flat or None

def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer, with rendered symbology."""
    lid = doc.add_geojson_layer(data=json.loads(gdf.to_json()), name=name,
                                symbology=symbology, **kwargs)
    flat = _flat_style(symbology, gdf)
    if flat:
        layer = doc._layers.get(lid)
        layer["parameters"]["color"] = flat
        doc._layers[lid] = layer
    return lid

In [5]:
from jupytergis_lab.notebook.symbology import constant

doc = new_map(stops_gdf, zoom=12)
add_gdf(doc, routes_gdf, "routes", symbology=[[constant("#2563eb").encoding("stroke")]])
add_gdf(doc, stops_gdf, "stops", symbology=[[constant("#111827").encoding("fill")]])
doc

C:\Users\Riz\AppData\Local\Temp\ipykernel_14416\3769270526.py:24: UserWarning: The JupyterGIS Python API is better experienced in the xeus-python kernel which supports awaiting comm messages
  doc = GISDocument(**kwargs)


## Where to go from here

With the transit database in place you can build a **TransitGraph**
(`aequilibrae.transit.TransitGraphBuilder`) and run schedule-based transit
assignment and skimming — see the *public transport assignment* example in the
AequilibraE documentation for the full workflow (hyperpath / optimal-strategies
assignment).


In [6]:
project.close()

---
**Next:** [08 — A full forecasting workflow](08_full_model_workflow.ipynb) ties
notebooks 03-05 together into a base-year/future-year model.
